# METU

In [25]:
from pathlib import Path


ROOT_DIR = Path(
    "/mnt/data/Public/Public/colorectum/METU/metu_cctgs"
)  # both annotations and slides

In [2]:
import os

import mlflow


os.environ["MLFLOW_TRACKING_USERNAME"] = "..."
os.environ["MLFLOW_TRACKING_PASSWORD"] = "..."
mlflow.set_tracking_uri("https://mlflow.rationai.cloud.e-infra.cz/")

/home/jovyan/carcinoma-binary-classification-methods/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Metadata Table

In [3]:
all_slides = list(ROOT_DIR.rglob("*.svs"))
len(all_slides)

104

In [4]:
import pandas as pd


df = pd.DataFrame(
    {"slide_path": all_slides, "carcinoma": [True for i in range(len(all_slides))]}
)

In [5]:
df.head()

,slide_path,carcinoma
0,/mnt/data/Public/Public/colorectum/METU/metu_c...,True
1,/mnt/data/Public/Public/colorectum/METU/metu_c...,True
2,/mnt/data/Public/Public/colorectum/METU/metu_c...,True
3,/mnt/data/Public/Public/colorectum/METU/metu_c...,True
4,/mnt/data/Public/Public/colorectum/METU/metu_c...,True


In [6]:
df.to_csv("./metu_full.csv", index=False)
mlflow.set_experiment("Colorectal cancer")

with mlflow.start_run(run_name="METU Metadata"):
    mlflow.log_artifact("./metu_full.csv")

🏃 View run METU Metadata at: https://mlflow.rationai.cloud.e-infra.cz/#/experiments/172/runs/ab887aab130e4d4b840df0eb219ebb04
🧪 View experiment at: https://mlflow.rationai.cloud.e-infra.cz/#/experiments/172


## Annotations

In [7]:
import json
from collections import Counter
from pathlib import Path
from xml.etree import ElementTree as ET


def scan_annotation_groups(annotation_dir: str | Path) -> Counter[str]:
    annotation_dir = Path(annotation_dir)
    group_counts: Counter[str] = Counter()

    for path in annotation_dir.rglob("*.xml"):
        root = ET.parse(path).getroot()
        for region in root.findall(".//Annotation"):
            group = region.get("PartOfGroup")
            if group is not None:
                group_counts[group] += 1

    for path in annotation_dir.rglob("*.geojson"):
        with open(path) as f:
            data = json.load(f)
        features = data if isinstance(data, list) else data["features"]
        for feature in features:
            classification = feature.get("properties", {}).get("classification", {})
            name = classification.get("name")
            if name is not None:
                group_counts[name] += 1
            else:
                group_counts["<UNCLASSIFIED>"] += 1

    return group_counts


counts = scan_annotation_groups(ROOT_DIR)

In [8]:
counts

Counter({'T-G2': 155,
         'T-G1': 126,
         'MP': 82,
         'T-G3': 78,
         'Normal mucosa': 70,
         'AT': 69,
         'LP': 21,
         'LVI': 21,
         'Stroma': 6,
         '<UNCLASSIFIED>': 2})

In [10]:
all_annots = list(ROOT_DIR.rglob("*.geojson"))
len(all_annots)

87

These were created after the initial exploration:

In [18]:
annot_masks = Path(
    mlflow.artifacts.download_artifacts(
        "mlflow-artifacts:/172/16a8707785db4bd29af80559efb9c4b0/artifacts/annotation_masks"
    )
)

All positive ?

In [19]:
n_pos = 0
for path in df["slide_path"]:
    stem = Path(path).stem

    g1 = annot_masks / "tumor_grade_1" / f"{stem}.tiff"
    g2 = annot_masks / "tumor_grade_2" / f"{stem}.tiff"
    g3 = annot_masks / "tumor_grade_3" / f"{stem}.tiff"

    if g1.exists() or g2.exists() or g3.exists():
        n_pos += 1

n_pos

87

In [ ]:
slides_with_annots = {path.stem for path in annot_masks.rglob("*.tiff")}
len(slides_with_annots)

87

In [21]:
test_slides = [
    path for path in df["slide_path"] if Path(path).stem not in slides_with_annots
]
len(test_slides)

17

## Slides

In [11]:
import openslide


mpps = set()
level_counts = set()
downsamples = set()
physical_sizes = set()

weird_mpp_slides = set()

for path in df["slide_path"]:
    slide = openslide.OpenSlide(path)
    mpp_x = float(slide.properties.get(openslide.PROPERTY_NAME_MPP_X))
    mpp_y = float(slide.properties.get(openslide.PROPERTY_NAME_MPP_Y))

    if mpp_x > 0.4:
        weird_mpp_slides.add(Path(path).stem)

    mpps.add((mpp_x, mpp_y))
    level_counts.add(slide.level_count)
    downsamples.add(slide.level_downsamples)

    slide.close()

In [12]:
mpps

{(0.2524, 0.2524), (0.2528, 0.2528), (0.5035, 0.5035)}

In [13]:
level_counts

{3, 4}

In [ ]:
downsamples

Filtered df logging - Only slides with annotations and mpp of 0.25 at level 0

In [24]:
df["stem"] = df["slide_path"].map(lambda x: Path(x).stem)
filtered_df = df[
    (~df["stem"].isin(weird_mpp_slides)) & df["stem"].isin(slides_with_annots)
]

filtered_df.to_csv("./metu_valid.csv", index=False)
mlflow.set_experiment("Colorectal cancer")

with mlflow.start_run(run_name="METU (Valid Subset) Metadata"):
    mlflow.log_artifact("./metu_valid.csv")

🏃 View run METU (Valid Subset) Metadata at: https://mlflow.rationai.cloud.e-infra.cz/#/experiments/172/runs/0c8b1e47432246b1bd954b2c672024d1
🧪 View experiment at: https://mlflow.rationai.cloud.e-infra.cz/#/experiments/172


## Rescaled Slides

We need x2 downsample (we run our pre-processing script based on this finding and here we gather the results):

In [35]:
loc = Path("/mnt/data/Public/Public/colorectum/METU/x2")

mpps = set()
level_counts = set()
downsamples = set()
physical_sizes = set()

for path in loc.glob("*.tiff"):
    slide = openslide.OpenSlide(path)
    mpp_x = float(slide.properties.get(openslide.PROPERTY_NAME_MPP_X))
    mpp_y = float(slide.properties.get(openslide.PROPERTY_NAME_MPP_Y))

    mpps.add((mpp_x, mpp_y))
    level_counts.add(slide.level_count)
    downsamples.add(slide.level_downsamples)

    slide.close()

In [36]:
mpps

{(0.5056000139040003, 0.5056000139040003)}

In [37]:
level_counts

{7, 8, 9}

In [ ]:
downsamples

Create metadata table pointing to these:

In [38]:
stem_to_rescaled_path = {file.stem: file for file in loc.glob("*.tiff")}
filtered_rescaled_df = filtered_df.copy()
filtered_rescaled_df["slide_path"] = filtered_rescaled_df["stem"].map(
    stem_to_rescaled_path
)

In [39]:
filtered_rescaled_df["slide_path"]

0      /mnt/data/Public/Public/colorectum/METU/x2/100...
1      /mnt/data/Public/Public/colorectum/METU/x2/920...
2      /mnt/data/Public/Public/colorectum/METU/x2/293...
3      /mnt/data/Public/Public/colorectum/METU/x2/b-6...
4      /mnt/data/Public/Public/colorectum/METU/x2/722...
                             ...                        
95     /mnt/data/Public/Public/colorectum/METU/x2/100...
97     /mnt/data/Public/Public/colorectum/METU/x2/280...
100    /mnt/data/Public/Public/colorectum/METU/x2/162...
102    /mnt/data/Public/Public/colorectum/METU/x2/b-2...
103    /mnt/data/Public/Public/colorectum/METU/x2/b-0...
Name: slide_path, Length: 77, dtype: object

In [40]:
filtered_rescaled_df.to_csv("./metu_rescaled_valid.csv", index=False)
mlflow.set_experiment("Colorectal cancer")

with mlflow.start_run(run_name="METU (Valid & Rescaled Subset) Metadata"):
    mlflow.log_artifact("./metu_rescaled_valid.csv")

🏃 View run METU (Valid & Rescaled Subset) Metadata at: https://mlflow.rationai.cloud.e-infra.cz/#/experiments/172/runs/a61e581fc2e84d1d971cfc1ec042597d
🧪 View experiment at: https://mlflow.rationai.cloud.e-infra.cz/#/experiments/172


## Local Visualization

In [11]:
!uv pip install --python ~/carcinoma-binary-classification-methods/.venv/bin/python xopat

Using Python 3.12.3 environment at: /home/jovyan/carcinoma-binary-classification-methods/.venv
Audited 1 package in 599ms


In [37]:
def get_masks(slide: pd.Series) -> list[tuple[str, str]]:
    stem = Path(slide["slide_path"]).stem
    occurences = list(annot_masks.rglob(f"*{stem}*"))
    res = []
    for occ in occurences:
        parts = str(occ).split("/")
        res.append((parts[-2], str(occ)))

    return res

In [45]:
import xopat


xopat.setup_jupyterhub("https://hub.rationai.cloud.e-infra.cz")

df["stem"] = df["slide_path"].map(lambda x: Path(x).stem)
slide = df[df["stem"] == "1009057"].iloc[0]

data_dir = "/"
slide_name = str(slide["slide_path"])
masks = get_masks(slide)

print(masks)

server = xopat.run_server(data_dir=data_dir)

session_config = {
    "data": [slide_name] + [path for name, path in masks],
    "background": [{"dataReference": 0}],
    "visualizations": [
        {
            "name": name,
            "shaders": {
                "heatmap": {
                    "type": "heatmap",
                    "dataReferences": [i + 1],
                    "params": {"color": "#FF0000", "opacity": 0.7},
                }
            },
        }
        for i, (name, path) in enumerate(masks)
    ],
}

xopat.display(server, session_config)

Configured for JupyterHub: https://hub.rationai.cloud.e-infra.cz/user/adam-dzadon/proxy/9001/
[('at', '/tmp/tmphfhjhx0m/annotation_masks/at/1009057.tiff'), ('lp', '/tmp/tmphfhjhx0m/annotation_masks/lp/1009057.tiff'), ('mp', '/tmp/tmphfhjhx0m/annotation_masks/mp/1009057.tiff'), ('normal_mucosa', '/tmp/tmphfhjhx0m/annotation_masks/normal_mucosa/1009057.tiff'), ('tumor_grade_2', '/tmp/tmphfhjhx0m/annotation_masks/tumor_grade_2/1009057.tiff')]
Stopping xOpat...
xOpat stopped.
Stopping WSI-Service...
WSI-Service stopped.
WSI-Service: using cached binary /home/jovyan/.xopat/wsi/wsi-v1.0.2/linux/wsi_service_binary (wsi-v1.0.2)
xOpat: using cached binary /home/jovyan/.xopat/xopat/xopat-v1.0.7/linux/xopat_binary (xopat-v1.0.7)
Starting WSI-Service...
WSI-Service is running.
Starting xOpat...
xOpat is running.
Servers running. Slides folder: /
